In [1]:
# Biogeme
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable, log, exp

# General python packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from pathlib import Path

# Pandas setting to show all columns when displaying a pandas dataframe
pd.set_option('display.max_columns', None)

In [4]:
# Create that path to the data file
data_path =  Path('/home/kiito/cycling_safety_svi/data/raw/cv_dcm.csv')
print(data_path)

/home/kiito/cycling_safety_svi/data/raw/cv_dcm.csv


In [5]:
# Load the data as a pandas dataframe
df = pd.read_csv(data_path)
df.head()

,ID,RID,SEQ,train,test,TL1,TT1,IMG1,TL2,TT2,IMG2,CHOICE
0,1,1,1,1,0,1,14,WE6MD43V.jpg,3,14,WE6JNFWP.jpg,1
1,2,1,6,1,0,3,11,WE6LRD7Z.jpg,3,14,WE6JIQH7.jpg,1
2,3,1,3,1,0,2,11,WE6LRB0P.jpg,2,8,WE6J8DRY.jpg,2
3,4,1,30,1,0,3,8,WE6OX763.jpg,3,11,WE6MCNP7.jpg,1
4,5,1,8,1,0,3,8,WE6KRJA1.jpg,2,8,WE6OX682.jpg,2


In [6]:
df['TL1'].value_counts()

TL1
2    3974
3    3680
1    3635
Name: count, dtype: int64

In [7]:
df['TT1'].value_counts()

TT1
11    4308
14    3591
8     3390
Name: count, dtype: int64

In [8]:
df['RID'].nunique()

752

In [9]:
pd.concat((df['IMG1'],df['IMG2']),axis=0).nunique()

6355

In [10]:
# db.Database takes as arguments (1) a name (string) and (2) a data set (pandas dataframe)
attributes = ['TL1','TT1','TL2','TT2','CHOICE']
biodata = db.Database('cycling_project_roos', df[attributes])

In [11]:
# Create the variables for biogeme
TL1, TT1, TL2, TT2, CHOICE = biodata.variables['TL1'], biodata.variables['TT1'], biodata.variables['TL2'], biodata.variables['TT2'], biodata.variables['CHOICE']

In [12]:
# Give a name to the model    
model_name = 'Linear-additive RUM-MNL'

B_TL = Beta('B_TL', 0, None, None, 0)
B_TT = Beta('B_TT', 0, None, None, 0)

# Utility functions
V1 = B_TL * TL1 + B_TT * TT1
V2 = B_TL * TL2 + B_TT * TT2    

In [13]:
def estimate_mnl(V1,V2,CHOICE,database,name):
    
    # Create a dictionary to list the utility functions with the numbering of alternatives
    V = {1: V1, 2: V2}
        
    # Create a dictionary called av to describe the availability conditions of each alternative, where 1 indicates that the alternative is available, and 0 indicates that the alternative is not available.
    # This shows that all alternatives were available to all respondents. 
    av = {1: 1, 2: 1} 

    # Define the choice model: The function models.logit() computes the MNL choice probabilities of the chosen alternative given the V. 
    prob = models.logit(V, av, CHOICE)

    # Define the log-likelihood   
    LL = log(prob)
   
    # Create the Biogeme object containing the object database and the formula for the contribution to the log-likelihood of each row using the following syntax:
    biogeme = bio.BIOGEME(database, LL)
    
    # The following syntax passes the name of the model:
    biogeme.modelName = name

    # Some object settings regaridng whether to save the results and outputs 
    biogeme.generate_pickle = False
    biogeme.generate_html = False
    biogeme.save_iterations = False
    

    # Syntax to calculate the null log-likelihood. The null-log-likelihood is used to compute the rho-square 
    biogeme.calculate_null_loglikelihood(av)

    # This line starts the estimation and returns the results object.
    results = biogeme.estimate()
     
    return results

In [14]:
# Estimate the model
results_MNL = estimate_mnl(V1,V2,CHOICE,biodata,model_name)

In [15]:
# Print the estimation statistics
print(results_MNL.short_summary())

# Get the model parameters in a pandas table and  print it
beta_hat_MNL = results_MNL.get_estimated_parameters()
statistics_MNL = results_MNL.get_general_statistics()
print(beta_hat_MNL)

# Store the LL of the MNL model for later use
LL_MNL = results_MNL.data.logLike

Results for model Linear-additive RUM-MNL
Nbr of parameters:		2
Sample size:			11289
Excluded data:			0
Null log likelihood:		-7824.939
Final log likelihood:		-7594.536
Likelihood ratio test (null):		460.8055
Rho square (null):			0.0294
Rho bar square (null):			0.0292
Akaike Information Criterion:	15193.07
Bayesian Information Criterion:	15207.73

         Value  Rob. Std err  Rob. t-test  Rob. p-value
B_TL -0.233479      0.015299   -15.260765           0.0
B_TT -0.111844      0.007226   -15.478738           0.0


### Comparisons

In [16]:
# Cross-entrophy comparison
CE_MNL = -results_MNL.get_general_statistics()['Final log likelihood'][0]/df.shape[0]
CE_CVDCM = 0.636
print(f'The CE of the MNL model is:\t{CE_MNL:0.3f}\nthe CE of the CVDCM model is:\t{CE_CVDCM}')

# Log-likelihood comparison
LL_CVDCM = -0.636 * df.shape[0]
print(f'\nThe LL of the MNL model is:\t{LL_MNL:0.2f}\nthe LL of the CVDCM model is:\t{LL_CVDCM:0.2f}')

The CE of the MNL model is:	0.673
the CE of the CVDCM model is:	0.636

The LL of the MNL model is:	-7594.54
the LL of the CVDCM model is:	-7179.80


### Analysis on the test set